# 레이저 그리드 품질검측 — 코랩

레이저 격자 이미지를 넣으면 **엑셀 조서 하나**가 나옵니다.

| 입력 | 필수? | 없으면 |
|---|---|---|
| 레이저 이미지 | **필수** | — |
| `camera_params.json` | 선택 | 사양 프로파일 값을 씁니다 |
| IMU | 선택 | **장비가 똑바로 서 있다고 가정**합니다 |
| 정답값 `cast_pixels.json` | 선택 | 검출 정확도·깊이 오차를 못 냅니다 |
| 장면 사진 (레이저 OFF) | 선택 | 레이저 이미지에서 선을 지워 배경으로 씁니다 |


## 1. 설치

In [ ]:
import os
if os.path.isdir('laser_grid/.git'):
    !git -C laser_grid pull -q          # 두 번째부터는 최신본만 당긴다
else:
    !git clone -q https://github.com/znlsl10-rgb/laser_grid.git
%cd /content/laser_grid
!pip install -q -r requirements.txt
!apt-get -qq install -y fonts-nanum > /dev/null 2>&1   # 3D 그림의 한글 라벨
!git log --oneline -1
print('준비 완료')


## 2. 촬영 점검 — 돌리기 전에 먼저

실장비 촬영이 오면 **이걸 먼저** 돌리세요. `hardware.py` 는 저장소 없이도
혼자 도는 파일이라, 장비 업체에 그 파일 하나만 보내도 됩니다.

세 가지를 봅니다.

1. **파일이 있는가** — 빠진 것이 있으면 그 자리에서 막습니다.
2. **이미지가 쓸 만한가** — 저장 형식·노출·선폭·채널 분리.
3. **사양이 이미지와 맞는가** ← 핵심.
   `camera_params.json` 에 적힌 초점거리를 믿지 않고 **이미지에서 직접 재어**
   견줍니다. 선 사이 간격은 f 와 발사각만으로 정해지고 거리·기선과 무관하기
   때문입니다. 굴림각도 이미지에서 재서 대조합니다.

파일은 다 있는데 숫자가 다른 장비의 것이면, 검측은 그대로 돌아가고 결과만
조용히 틀립니다. 그 상태를 여기서 잡습니다.

먼저 저장소에 든 예제로 어떻게 나오는지 봅니다.

In [ ]:
import hardware
hardware.colab('samples/example_capture')     # 전체 구성 예제
# hardware.colab('samples/minimal_capture')   # 필수 2개만 있는 최소 구성
# hardware.colab('samples/rolled_capture')    # 격자를 20° 굴린 구성


### 내 촬영을 점검하기

아래 셀을 돌리면 업로드 창이 뜹니다. **zip 하나**를 올려도 되고,
`laser_on.png` · `camera_params.json` 같은 **파일 여러 개를 한꺼번에** 골라
올려도 됩니다. zip 안에 촬영 폴더가 여럿이면 각각 따로 점검합니다.

| 파일 | 필수? | 없으면 |
|---|:---:|---|
| `laser_on.png` | **필수** | 돌릴 수 없습니다 |
| `camera_params.json` | **필수** | 깊이가 통째로 배율만큼 틀립니다 |
| `laser_off.png` | 권장 | 햇빛 드는 현장에서 선을 놓칠 수 있습니다 |
| `imu.json` | 권장 | 장비가 똑바로 섰다고 가정하고 판정을 참고값으로 낮춥니다 |
| `truth.json` | 선택 | 검출 정확도·깊이 오차를 못 냅니다 (현장에는 없어도 됩니다) |

점검이 끝나면 `점검결과.json` 이 자동으로 내려받아집니다 — 업체는 이 파일을
그대로 회신하면 됩니다.

In [ ]:
import hardware
chk = hardware.colab()      # 업로드 → 압축 해제 → 점검까지 한 번에


판정은 세 가지입니다.

* **검측 가능** — 이대로 쓰면 됩니다.
* **사양과 이미지가 어긋난다** — 파일은 다 있지만 숫자가 이 이미지의 것이
  아닙니다. `[불일치]` 항목에 무엇이 몇 % 다른지 나옵니다.
* **검측 불가** — 필수 항목이 빠졌습니다.

`[경고]` 는 돌아가지만 정확도나 판정 등급에 영향을 줍니다.

> **기선(baseline)은 사진 한 장으로 검증할 수 없습니다.** 깊이 식에 기선 b 와
> 거리 Z 가 `b/Z` 로만 들어와서, '기선 2배'와 '거리 2배'가 구분되지 않습니다.
> 검증하려면 **거리를 자로 잰 촬영**을 한 벌 넣고 `camera_params.json` 에
> `"측정거리_m": 1.50` 처럼 적어 주세요. 그러면 점검기가 대조합니다.

업체에 넘길 규약서는 `docs/하드웨어_인터페이스_규약.md` 입니다.

## 3. 파일 올리기

왼쪽 파일 탭에 끌어다 놓아도 되고, 아래 셀로 올려도 됩니다.

In [ ]:
from google.colab import files
up = files.upload()          # 이미지 (+ 있으면 json 들)
print(list(up))

## 4. 실행

`image` 와 `params` 가 **둘 다 필수**입니다. 규약을 만족하는 촬영이 들어온다고
전제하고 최종 결과를 냅니다 — 사양 없이 돌리면 깊이가 배율만큼 틀리므로
아예 받지 않습니다. 위 2번 점검을 먼저 통과시키세요.

In [ ]:
import os
from run_pipeline import run

def find(name):
    """파일 탭에 끌어다 놓으면 /content 에, 업로드 셀로 올리면 여기에 온다.
    둘 다 찾아본다. 없으면 None — 선택 입력이면 그대로 넘어간다."""
    if not name:
        return None
    for c in (name, f'/content/{name}', f'/content/laser_grid/{name}'):
        if os.path.exists(c):
            return c
    print(f'  [없음] {name} — 이 입력 없이 진행합니다')
    return None

res = run(
    image  = find('CAST.png'),              # 필수 — 레이저 격자 이미지
    params = find('camera_params.json'),    # 선택 — 카메라 사양
    truth  = find('cast_pixels.json'),      # 선택 — 정답값
    imu    = None,                          # 선택 — 없으면 똑바로 섰다고 가정
    scene_image = find('CAM.png'),          # 선택 — 레이저 OFF 사진
    out    = '/content/결과/',              # 폴더를 줘도 되고 .xlsx 를 줘도 된다
)

print('조서:', res['xlsx'])


### IMU 를 줄 때

셋 중 아무 형식이나 됩니다.

```python
imu = {'pitch_deg': 34.0, 'roll_deg': 0.0}   # 아래로 숙인 각 / 광축 둘레 회전
imu = {'gravity': [0, 0.83, 0.56]}          # 조사기 좌표계 중력 벡터
imu = {'accel':   [0, -0.83, -0.56]}        # 정지 상태 가속도계 읽음
```

**IMU 를 안 주면 장비가 똑바로 서 있다고 가정합니다.** 수직도는 면 법선과 중력의 사잇각이라, 장비가 실제로 1° 기울어 있었다면 판정도 1° 틀립니다 — 허용치가 ±0.5° 인 것을 생각하면 작지 않습니다.

## 5. 결과 보기

In [ ]:
import pandas as pd
pd.read_excel(res['xlsx'], sheet_name='6.검측결과')

In [ ]:
from IPython.display import Image, display
for k in ('3D점군', '세그멘테이션', '선검출대조'):
    p = res['images'].get(k)
    if p:
        print(k); display(Image(p, width=900))

### 3D 점군을 다시 그리기 — 각도·좌표계·색칠 기준을 바꿔서

조서에 들어간 3D 그림은 `plot_points3d.py` 가 그린 것입니다. 같은 결과로
시점이나 색칠 기준만 바꿔 다시 그릴 수 있습니다.

* `by='member'` 부재 하나하나 (기본, 엑셀 9번 시트의 부재 번호와 같은 번호)
* `by='class'` 종류별로 (벽 / 바닥 / 동바리)
* `by='line'` V·H 선 계열 (파랑 / 빨강)
* `frame='gravity'` 중력 정렬(가로·깊이·높이) / `frame='camera'` 조사기 좌표
* `views='iso'` 큰 등각 하나 / `views='quad'` 등각+평면도+정면도+측면도


In [ ]:
import plot_points3d as P3
from IPython.display import Image, display

P3.save_pointcloud_mpl('/content/점군_등각.png', res['result'], res['g_hat'],
                       by='member', frame='gravity', views='iso',
                       elev=22, azim=-58, title='3D 점군 (부재별)')
display(Image('/content/점군_등각.png', width=900))

# 레퍼런스와 같은 V·H 색칠로 보고 싶으면
P3.save_pointcloud_mpl('/content/점군_선계열.png', res['result'], res['g_hat'],
                       by='line', views='iso', title='3D 점군 (V·H 선 계열)')
display(Image('/content/점군_선계열.png', width=900))


## 6. 내려받기

In [ ]:
from google.colab import files
files.download(res['xlsx'])                    # 엑셀 조서
# files.download(res['images']['3D점군'])       # 3D 점군 그림
# files.download(res['images']['세그멘테이션'])   # 세그멘테이션 그림


---
## 엑셀 조서 구성

| 시트 | 내용 |
|---|---|
| 1.요약 | 입력·가정·결과 한눈에 |
| 2.설계값 | 사양과 그 출처 등급 |
| **3.선검출(1단계)** | 선별 검출 결과 + 정답 대조 + 대조 그림 |
| **4.깊이검증(2단계)** | 화소→3D 깊이, 정답이 있으면 mm 오차표 |
| **5.세그멘테이션(3단계)** | 색↔부재 대응 + 그림 |
| **6.검측결과** | 부재별 수직도·수평도·평활도 판정 |
| 7~8 | 평활도 근거 · 요철 위치 + 그림 |
| **9.3D좌표(부재별)** | 3D 산점도(부재별 색) + 점수·중심·크기 |
| 10.3D좌표(점목록) | 부재 구분이 붙은 좌표 목록 |
| 11.유의사항 | 이 값을 어디까지 믿을 수 있는가 |

## 판정에 '판정보류' 가 나오면

부재에 세로 격자선이 **한 줄만** 걸린 경우입니다. 한 줄에서 나온 3D 점은
그 레이저 평면 안에 놓이므로, 위·아래 화소의 깊이 차이로 잡히는 것은
**카메라 쪽으로 넘어진 성분**뿐입니다. 옆으로 넘어진 성분은 깊이를 전혀
바꾸지 않아 보이지 않습니다.

그래서 잰 값은 참값의 **하한**이고, 판정이 한쪽으로만 성립합니다.

| 잰 값 | 판정 | 왜 |
|---|---|---|
| 허용치 초과 | **기준초과 (확정)** | 참값은 하한보다 크니 뒤집힐 수 없다 |
| 허용치 이내 | **판정보류(단면 미분해)** | 못 본 성분이 넘었을 수 있다 |

부재 뒤에 배경이 있으면 가로선 끊김(그림자)에서 옆 성분을 되찾아 정상
판정으로 돌아옵니다. 근본 해결은 부재에 세로선이 2줄 이상 걸리도록
더 가까이서 찍거나 격자를 조밀하게 하는 것입니다.

## 정확도에 대해

선검출의 한계는 알고리즘이 아니라 **입력 이미지**가 정합니다. 선이
안티에일리어싱 없이 이진(0/255)으로 그려져 있으면 선 중심이 0.5px 격자에
갇혀, 어떤 추정기를 써도 σ = 1/√12 = **0.289px** 아래로 못 내려갑니다.
파이프라인이 이 하한을 자동으로 재서 조서에 적습니다.
